# Kirana — Training model pilihan ganda

Notebook ini menelusuri eksperimen yang sama dengan `train.py`. Pada kuis semi-adaptif, pemilih soal berbasis aturan menentukan 6 soal lanjutan setelah 9 soal inti. Model yang dilatih di sini mendiagnosis pola setelah 15 soal selesai.

Untuk melihat hasil yang sudah tersedia, jalankan sel **Persiapan** dan **Buka laporan yang sudah ada**. Mulai dari **Simulasi ulang** jika ingin melatih ulang sendiri.

## 1. Persiapan

In [ ]:
import json
from pathlib import Path
import sys

from IPython.display import Markdown, display

candidates = (Path.cwd(), *Path.cwd().parents)
repo = next((path for parent in candidates
             for path in (parent, parent / "Source Code" / "Kirana")
             if (path / "ml" / "training.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Buka notebook dari folder proyek Kirana.")

sys.path.insert(0, str(repo))
from backend.src.quiz import CATALOG
from ml.diagnosis import METHODS, fit
from ml.paths import ARTIFACT, DATA_DIR, EVALUATION_REPORT
from ml.training import collect, export_and_verify, metrics

MODES = ("fixed", "adaptive")
SEED = 42

def show_summary(results):
    lines = [
        "| Paket | Operasi | Metode | Macro-F1 | Dugaan keliru | Kedua pola ditemukan | Tanpa dugaan |",
        "|---|---|---|---:|---:|---:|---:|",
    ]
    for mode in MODES:
        for operation in CATALOG:
            for method in METHODS:
                result = results[mode, operation, method]
                lines.append(
                    f"| {mode} | {operation} | {method} | {result['macro_f1']:.4f} | "
                    f"{result['false_pattern_calls']} | "
                    f"{result['mixed_both_found']} / {result['mixed_sessions']} | {result['abstained']} |"
                )
    display(Markdown("\n".join(lines)))

## 2. Buka laporan yang sudah ada

Sel ini hanya membaca metrik test sintetis dari `reports/evaluation.json`.

In [ ]:
report_path = EVALUATION_REPORT
if report_path.is_file():
    saved = json.loads(report_path.read_text(encoding="utf-8"))
    show_summary({
        (mode, operation, method): saved["results"][mode][operation]["test"][method]
        for mode in MODES for operation in CATALOG for method in METHODS
    })
else:
    print("Belum ada reports/evaluation.json. Jalankan simulasi dan evaluasi di bawah.")

## 3. Simulasi ulang

Bangun respons pilihan ganda untuk paket tetap dan semi-adaptif dengan bank soal dan simulator asli. Pertahankan split siswa dan soal dari data revisi. Dalam tiap mode, fitur diambil dari soal dan opsi yang benar-benar diberikan.

In [ ]:
samples, outcomes = collect(DATA_DIR, SEED)
print("Jumlah sesi per split, ditampilkan per paket")
for mode in MODES:
    for split in ("train", "validation", "test"):
        counts = {op: len(samples[mode, split, op][0]) for op in CATALOG}
        print(mode, split, counts)

## 4. Training dan validation

Fit regresi logistik multilabel pada `train`, lalu tetapkan ambang per pola untuk model dan baseline aturan pada `validation`.

In [ ]:
models = {
    (mode, operation): fit(operation, samples[mode, "train", operation],
                           samples[mode, "validation", operation])
    for mode in MODES for operation in CATALOG
}

def evaluate(split):
    return {
        (mode, operation, method): metrics(operation, model, samples[mode, split, operation], method)
        for (mode, operation), model in models.items()
        for method in METHODS
    }

validation = evaluate("validation")
show_summary(validation)

## 5. Evaluasi test

Gunakan ambang yang sudah ditetapkan. Ringkasan membandingkan kedua mode dan kedua metode pada siswa dan soal yang ditahan. Detail per pola tersedia dalam hasil di bawah. Angka menggambarkan simulasi.

In [ ]:
test = evaluate("test")
show_summary(test)
for operation in CATALOG:
    print(f"Rincian semi-adaptif / logistik / {operation}")
    for pattern, result in test["adaptive", operation, "logistic"]["by_pattern"].items():
        print(f"{pattern}  precision={result['precision']:.3f}  "
              f"recall={result['recall']:.3f}  f1={result['f1']:.3f}  support={result['support']}")

## 6. Ekspor opsional

Ubah `EXPORT_MODEL` ke `True` untuk mengganti artefak model pilihan ganda lokal dengan hasil training ini. Fungsi ekspor memuat ulang model dan membandingkan hasil prediksinya. Setelah ekspor, jalankan ulang Flask untuk memuat model.

In [ ]:
EXPORT_MODEL = False

if EXPORT_MODEL:
    adaptive_models = {operation: models["adaptive", operation] for operation in CATALOG}
    export_and_verify(ARTIFACT, adaptive_models, samples)
    print(f"Model tersimpan dan lolos cek muat ulang: {ARTIFACT}")
else:
    print("Ekspor nonaktif.")